In [ ]:
import os
import pandas as pd
import re # لاستخدام التعابير النمطية

def extract_final_answer(raw_pred_text):
    """
    تستخلص هذه الدالة الإجابة النهائية من نص 'raw_preds'.
    تبحث عن النمط "الإجابة النهائية هي [X]" أو "الإجابة النهائية هي (X)"
    حيث X هو حرف واحد.
    """
    if pd.isna(raw_pred_text): # التحقق مما إذا كانت القيمة NaN
        return pd.NA # إرجاع قيمة NaN الخاصة بـ pandas إذا كان الإدخال NaN

    # تحويل المدخلات إلى سلسلة نصية لضمان عمل التعابير النمطية بشكل صحيح
    text_to_search = str(raw_pred_text)

    # التعبير النمطي للبحث عن الإجابة النهائية.
    # يبحث عن "الإجابة النهائية هي" متبوعة بمسافات اختيارية
    # ثم حرف واحد داخل أقواس مربعة أو دائرية.
    # (?:...) هي مجموعة غير لاقطة، مما يسمح لنا باستخدام "أو" |
    match = re.search(r"الإجابة النهائية هي\s*\[(.)\]|الإجابة النهائية هي\s*\((.)\)", text_to_search)

    if match:
        # group(1) سيكون للحرف داخل الأقواس المربعة، group(2) للحرف داخل الأقواس الدائرية
        # أحدهما سيكون None والآخر سيحتوي على الحرف.
        if match.group(1):
            return match.group(1)
        elif match.group(2):
            return match.group(2)
    else:
        # إذا لم يتم العثور على النمط المحدد، حاول البحث عن نمط أعم للإجابة بين قوسين (مربعة أو دائرية)
        # هذا للتعامل مع الحالات التي قد لا تحتوي على "الإجابة النهائية هي" بالضبط
        # ابحث عن آخر تطابق لأنه قد يكون هناك عدة أقواس
        all_fallback_matches = re.findall(r"\[([أ-يa-zA-Z])\]|\(([أ-يa-zA-Z])\)", text_to_search)
        if all_fallback_matches:
            # re.findall مع مجموعات متعددة (بسبب الـ |) سيعيد قائمة من الـ tuples
            # مثال: [('', 'أ'), ('ب', '')]
            # نحتاج إلى أخذ آخر tuple غير فارغ
            last_match_tuple = all_fallback_matches[-1]
            # إرجاع العنصر غير الفارغ من الـ tuple
            if last_match_tuple[0]: # إذا كان الحرف في القوس المربع
                return last_match_tuple[0]
            elif last_match_tuple[1]: # إذا كان الحرف في القوس الدائري
                return last_match_tuple[1]
        return pd.NA # إرجاع قيمة NaN الخاصة بـ pandas إذا لم يتم العثور على النمط

def process_csv_files(data_folder="data", output_folder="output"):
    """
    تقرأ هذه الدالة ملفات CSV من 'data_folder'، وتستخلص الإجابات،
    وتحفظ الملفات المعدلة في 'output_folder'.
    """
    # التأكد من وجود مجلد الإدخال
    if not os.path.isdir(data_folder):
        print(f"خطأ: لم يتم العثور على مجلد الإدخال '{data_folder}'. يرجى إنشاء المجلد ووضع ملفات CSV فيه.")
        return

    # إنشاء مجلد الإخراج إذا لم يكن موجودًا
    os.makedirs(output_folder, exist_ok=True)
    print(f"سيتم حفظ الملفات المعالجة في المجلد: '{output_folder}'")

    # المرور على جميع الملفات في مجلد البيانات
    for filename in os.listdir(data_folder):
        if filename.endswith(".csv"):
            input_file_path = os.path.join(data_folder, filename)
            output_file_path = os.path.join(output_folder, filename)

            print(f"جاري معالجة الملف: {input_file_path}...")

            try:
                # قراءة ملف CSV
                # تحديد الفاصلة بشكل صريح وتجربة محركات مختلفة إذا لزم الأمر
                df = pd.read_csv(input_file_path, sep=',', engine='python', encoding='utf-8')

                # التأكد من وجود عمود 'raw_preds'
                if 'raw_preds' not in df.columns:
                    print(f"  تحذير: لم يتم العثور على عمود 'raw_preds' في الملف {filename}. يتم تخطي هذا الملف.")
                    continue

                # تطبيق دالة الاستخلاص على عمود 'raw_preds' لملء عمود 'preds'
                # سيتم إنشاء عمود 'preds' إذا لم يكن موجودًا، أو سيتم الكتابة فوقه إذا كان موجودًا.
                df['preds'] = df['raw_preds'].apply(extract_final_answer)

                # حفظ DataFrame المعدل في ملف CSV جديد في مجلد الإخراج
                # التأكد من استخدام ترميز utf-8 للحفاظ على الأحرف العربية
                df.to_csv(output_file_path, index=False, encoding='utf-8')
                print(f"  تم حفظ الملف المعالج: {output_file_path}")

            except Exception as e:
                print(f"  حدث خطأ أثناء معالجة الملف {filename}: {e}")

if __name__ == "__main__":
    # اسم مجلد الإدخال الذي يحتوي على ملفات CSV
    data_directory = "data"
    # اسم مجلد الإخراج حيث سيتم حفظ النتائج
    output_directory = "output_preds"

    process_csv_files(data_folder=data_directory, output_folder=output_directory)
    print("اكتملت المعالجة.")



سيتم حفظ الملفات المعالجة في المجلد: 'output_preds'
جاري معالجة الملف: data/result_prompt_ar_alpa_arcot__openai_gpt-4o-mini-2024-07-18.csv...
  تم حفظ الملف المعالج: output_preds/result_prompt_ar_alpa_arcot__openai_gpt-4o-mini-2024-07-18.csv
اكتملت المعالجة.
